In [ ]:
!pip install -q openai datasets scikit-learn tqdm pandas

In [ ]:
import os
from openai import OpenAI

try:
    from google.colab import userdata
    API_KEY = userdata.get("OPENAI_API_KEY")

    if not API_KEY:
        raise ValueError("Key not found.")

    os.environ["OPENAI_API_KEY"] = API_KEY

except Exception as err:
    print({err})
    raise

try:
    client = OpenAI(api_key=API_KEY)

    test = client.chat.completions.create(
        model="gpt-5",
        messages=[{"role": "user", "content": "What is the climate in Norway?"}],
    )
    print(test.choices[0].message.content)
    print(f"Tokens: {test.usage.total_tokens}")

except Exception as err:
    print({err})
    raise

In [ ]:
import pandas as pd
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

dataset_original = load_dataset("ailsntua/QEvasion")
test_df = dataset_original['test'].to_pandas()

response_to_label_map = {
    "clear reply": "Clear Reply",
    "clear non-reply": "Clear Non-Reply",
    "clear non reply": "Clear Non-Reply",
    "ambivalent": "Ambivalent",
}

In [ ]:
TAXONOMY_TEXT = """
1. Clear Reply - The information requested is explicitly stated (in the requested form)
2. Clear Non-Reply - The information requested is not given at all due to ignorance, need for clarification or declining to answer
3. Ambivalent - The information requested is given in an incomplete way e.g. the answer is too general, partial, implicit, dodging or deflection
"""

SYSTEM_MESSAGE = f"""You are an expert in analyzing political discourse and question evasion patterns.
Your task is to classify the type of response provided by an interviewee based on a segment of the interview using the following taxonomy:

{TAXONOMY_TEXT}

In the following examples, you will see 8 demonstration cases for different scenarios.
Each demonstration consists of 2 messages: one user message with the question and answer, followed by one assistant message showing the step-by-step reasoning and final classification."""

FEW_SHOT_EXAMPLES = [
    {
        "role": "user",
        "content": """### Part of the interview ###
The interviewer is asking various questions about political views and policies.

### Question ###
Do you have your own views about PR at Westminster don't you?

### Answer ###
I do."""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks: "Do you have your own views about PR at Westminster?"
2. The answer is: "I do."
3. This directly confirms having views, explicitly answering the yes/no question.
4. The information requested is explicitly stated in the requested form.

Taxonomy code: Clear Reply"""
    },

    {
        "role": "user",
        "content": """### Part of the interview ###
The interviewer is asking about evening plans.

### Question ###
Are you going to watch television?

### Answer ###
What else is there to do?"""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks: "Are you going to watch television?"
2. The answer is: "What else is there to do?"
3. This rhetorical question suggests they will watch TV, but doesn't explicitly state it.
4. The information is given in an incomplete, implicit way.

Taxonomy code: Ambivalent"""
    },

    {
        "role": "user",
        "content": """### Part of the interview ###
The interviewer is asking personal preference questions.

### Question ###
Do you like my new dress?

### Answer ###
We are late."""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks: "Do you like my new dress?"
2. The answer is: "We are late."
3. The interviewee completely ignores the question about the dress.
4. This is dodging - the information is given incompletely (not at all, in fact).

Taxonomy code: Ambivalent"""
    },

    {
        "role": "user",
        "content": """### Part of the interview ###
The interviewer is asking about a recent event.

### Question ###
Did you eat the last piece of pie?

### Answer ###
I have to admit that this was a great recipe, I always like it when there are chocolate chips in the dough."""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks: "Did you eat the last piece of pie?"
2. The answer talks about the recipe and chocolate chips, deflecting from whether they ate it.
3. This is deflection - starting on topic but shifting focus away from the actual question.
4. The information is given in an incomplete way.

Taxonomy code: Ambivalent"""
    },

    {
        "role": "user",
        "content": """### Part of the interview ###
The interviewer is asking about entertainment preferences.

### Question ###
Did you enjoy the film?

### Answer ###
The directing was great."""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks: "Did you enjoy the film?"
2. The answer only addresses the directing, which is one component of a film.
3. The interviewee doesn't address overall enjoyment - this is a partial answer.
4. The information is given in an incomplete way.

Taxonomy code: Ambivalent"""
    },

    {
        "role": "user",
        "content": """### Part of the interview ###
The interviewer is asking about favorite movies.

### Question ###
What's your favorite film?

### Answer ###
Fight Club, Filth, and Hereditary."""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks for ONE favorite film (singular).
2. The answer provides THREE films.
3. This makes the desired information unclear - the answer is too general.
4. The information is given in an incomplete way.

Taxonomy code: Ambivalent"""
    },

    {
        "role": "user",
        "content": """### Part of the interview ###
The interviewer is discussing political hypotheses.

### Question ###
The hypothesis I was discussing, wouldn't you regard that as a defeat?

### Answer ###
I am not going to prophesy what will happen."""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks for their view on whether something would be a defeat.
2. The answer is: "I am not going to prophesy what will happen."
3. The interviewee directly states they won't answer - declining to answer.
4. The information requested is not given at all.

Taxonomy code: Clear Non-Reply"""
    },

    {
        "role": "user",
        "content": """### Part of the interview ###
The interviewer is asking about government decisions.

### Question ###
On what precise date did the government order the refit of the HMAS Kanimbla in preparation for its forward deployment to a possible war against Iraq?

### Answer ###
I do not know that date. I will find out and let the House know."""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks for a specific date.
2. The answer is: "I do not know that date."
3. The interviewee admits they don't have the information - claims ignorance.
4. The information requested is not given due to ignorance.

Taxonomy code: Clear Non-Reply"""
    },

    {
        "role": "user",
        "content": """### Part of the interview ###
The interviewer is asking about financial decisions.

### Question ###
Was it your decision to release the fund?

### Answer ###
You mean the public fund?"""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks: "Was it your decision to release the fund?"
2. The answer is: "You mean the public fund?"
3. The interviewee doesn't provide any information, but asks for clarification.
4. The information requested is not given due to need for clarification.

Taxonomy code: Clear Non-Reply"""
    }
]

USER_PROMPT_TEMPLATE = """Now, you have to analyze this new case:

### Question ###
{question}

### Answer ###
{interview_answer}

Instructions for your analysis:
1. Think step by step about the relationship between the question and the answer.
2. Provide a brief explanation of your reasoning.
3. You MUST end your response with "Taxonomy code: [Label Name]" exactly as shown in the examples above.

Let's think step by step and provide your classification."""

In [ ]:
from tqdm import tqdm
import time

predictions = []
api_call_times = []

print(f"Starting GPT-5 prompting on {len(test_df)} test examples...\n")

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Processing"):
    question = row['question']
    interview_answer = row['interview_answer']

    user_prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        interview_answer=interview_answer
    )

    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE}
    ] + FEW_SHOT_EXAMPLES + [
        {"role": "user", "content": user_prompt}
    ]

    start_time = time.time()

    try:
        response = client.chat.completions.create(
            model="gpt-5",
            messages=messages,
        )

        api_call_times.append(time.time() - start_time)
        response_text = response.choices[0].message.content.strip()

        predicted_label = None
        if "taxonomy code:" in response_text.lower():
            parts = response_text.lower().split("taxonomy code:")
            if len(parts) > 1:
                candidate = parts[-1].strip().split('\n')[0].strip()
                predicted_label = response_to_label_map.get(candidate)

        if predicted_label is None:
            print(f"\nError at index {idx}: Label not found in response")
            print(f"Response: {response_text}")
            break

        predictions.append(predicted_label)

    except Exception as e:
        print(f"\nError at index {idx}: {e}")
        break

print(f"\n Prompting complete: {len(predictions)} predictions")
print(f"  Total time: {sum(api_call_times)/60:.1f} minutes\n")

results_df = test_df.copy()
results_df['predicted_label'] = predictions

output_file = "gpt5-fscot-clarity-test-predictions.csv"
results_df.to_csv(output_file, index=False)

In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score
import numpy as np

print("Subtask 1: Clarity Classification\n")

y_true_clarity = results_df['clarity_label'].tolist()
y_pred_clarity = predictions

accuracy_clarity = accuracy_score(y_true_clarity, y_pred_clarity)

clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]
macro_f1_clarity = f1_score(y_true_clarity, y_pred_clarity, average='macro', labels=clarity_labels, zero_division=0)

print(f"Accuracy: {accuracy_clarity:.4f}")
print(f"Macro F1: {macro_f1_clarity:.4f}")

print("\nPer-class metrics:")
print(classification_report(y_true_clarity, y_pred_clarity, labels=clarity_labels, zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_true_clarity, y_pred_clarity, labels=clarity_labels)
print(f"Labels order: {clarity_labels}")
print(cm)